In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/05 12:20:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/03/05 12:20:38 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/03/05 12:20:38 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/03/05 12:20:38 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [2]:
spark.version

'3.5.5'

In [3]:
!wget -O data/yellow_tripdata_2024-10.parquet https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

--2025-03-05 12:35:12--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.167.84.228, 3.167.84.127, 3.167.84.86, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.167.84.228|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64346071 (61M) [binary/octet-stream]
Saving to: 'data/yellow_tripdata_2024-10.parquet'

data/yellow_tripdat 100%[===================>]  61.36M  20.2MB/s    in 3.0s    

2025-03-05 12:35:15 (20.2 MB/s) - 'data/yellow_tripdata_2024-10.parquet' saved [64346071/64346071]



In [4]:
!wc -l data/yellow_tripdata_2024-10.parquet

  254534 data/yellow_tripdata_2024-10.parquet


In [5]:
from pyspark.sql import types

In [23]:
yellow_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.LongType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.LongType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("payment_type", types.LongType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

In [24]:
df_yellow_oct_2024 = spark.read \
    .option("header", "true") \
    .schema(yellow_schema) \
    .parquet('data/yellow_tripdata_2024-10.parquet')

In [21]:
# df_yellow_oct_2024 = spark.read \
#     .option("header", "true") \
#     .parquet('data/yellow_tripdata_2024-10.parquet')

In [22]:
# df_yellow_oct_2024.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [25]:
df_yellow_oct_2024 \
    .repartition(4) \
    .write.parquet('data/pq/yellow/2024/10', mode='overwrite')

In [26]:
from pyspark.sql import functions as F

In [28]:
df_pickup_oct_15 = df_yellow_oct_2024.select("tpep_pickup_datetime") \
    .filter(F.to_date(df_yellow_oct_2024.tpep_pickup_datetime) == "2024-10-15")

In [29]:
df_pickup_oct_15.count()

128909

In [34]:
df_yellow_oct_2024.registerTempTable('yellow')

/usr/local/Cellar/apache-spark/3.5.5/libexec/python/pyspark/sql/dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [35]:
df_trip_duration = spark.sql("""
SELECT
    tpep_pickup_datetime,
    tpep_dropoff_datetime,
    TIMESTAMPDIFF(HOUR, tpep_pickup_datetime, tpep_dropoff_datetime) as trip_duration
FROM
    yellow
ORDER BY
    trip_duration DESC
LIMIT 1
""")

In [36]:
df_trip_duration.show()

+--------------------+---------------------+-------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration|
+--------------------+---------------------+-------------+
| 2024-10-16 09:03:49|  2024-10-23 03:40:53|          162|
+--------------------+---------------------+-------------+



In [37]:
df_zones = spark.read.parquet('zones/')
df_zones.createOrReplaceTempView('zonelookup')
df_yellow_oct_2024.createOrReplaceTempView('yellowtaxi')

In [38]:
df_least_freq_zone = spark.sql("""
SELECT
    z.zone,
    COUNT (y.PULocationID) as pickup_count
FROM zonelookup z
JOIN yellowtaxi y ON y.PULocationID = z.LocationID
GROUP BY z.zone
ORDER BY pickup_count ASC
LIMIT 1
""")

In [39]:
df_least_freq_zone.show()

+--------------------+------------+
|                zone|pickup_count|
+--------------------+------------+
|Governor's Island...|           1|
+--------------------+------------+



In [40]:
spark.stop()